# Session 4 — Classification

*Machine Learning · Grau en Enginyeria Informàtica · Universitat de Barcelona · 2026-27*

> **Today's question.** A bank's model correctly classifies **97.2%** of its customers as "will default" or
> "will not default" on their credit card. Should the bank deploy it?
> **A)** Yes, 97% is excellent · **B)** It depends on which customers it gets wrong · **C)** No
>
> Write your answer down. We come back to it in Section 4.

**Contents**

1. Why not linear regression?
2. Logistic regression — model, interpretation and cross-entropy
3. Confounding — a coefficient depends on what else is in the model
4. Evaluating a classifier — from probabilities to decisions
5. ROC and Precision-Recall curves
6. More than two classes — softmax
7. Optional: LDA, a generative alternative
8. Wrap-up
9. Practice at home — a "5" detector

**How to use this notebook.** In class we run some cells live. A cell preceded by **Predict first** is a
question: commit to an answer *before* running it.

**Notation** (same as Session 3): $n$ observations, $p$ features, $\beta$ coefficients,
$p(x) = \Pr(Y = 1 \mid X = x)$ the predicted probability, $\sigma(z) = 1/(1 + e^{-z})$ the sigmoid.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.calibration import CalibrationDisplay
from sklearn.datasets import load_digits
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, precision_recall_curve, roc_curve,
                             accuracy_score, average_precision_score, balanced_accuracy_score,
                             brier_score_loss, confusion_matrix, f1_score, log_loss, make_scorer,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42  # every random step uses this seed: re-running gives identical numbers

BLUE, ORANGE, GREEN, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8a85"
plt.rcParams.update({
    "figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, GREEN, YELLOW]),
})

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
if tuple(int(v) for v in sklearn.__version__.split(".")[:2]) < (1, 5):
    print("WARNING: Section 4 needs scikit-learn >= 1.5. Run:  pip install -U scikit-learn")


def load_csv(local_path, url, **kwargs):
    """Read the course copy of a dataset; download it from `url` if it is not there."""
    path = Path(local_path)
    return pd.read_csv(path if path.exists() else url, **kwargs)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

**Default** (ISLR): 10,000 credit-card customers (simulated). `default` = did the customer default on their
debt? `balance` = average monthly balance on the card (\$), `income` = yearly income (\$), `student` = yes/no.

In [ ]:
raw = load_csv("dataset/classification/Default.csv",
               "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/ISLR/Default.csv",
               index_col=0)
# Encode the two yes/no columns as 0/1
df = raw.assign(default=(raw["default"] == "Yes").astype(int),
                student=(raw["student"] == "Yes").astype(int))

print(f"{len(df)} customers, {df['default'].sum()} defaults ({df['default'].mean():.1%})")
df.head()

## 1. Why not linear regression?

Code the response as $Y = 1$ (default) or $Y = 0$ and fit a straight line: $\hat y = \beta_0 + \beta_1\,\text{balance}$.

**Predict first.** What will this line predict for a customer with a balance of \$0?
**A)** exactly 0 · **B)** a small positive number · **C)** a negative number · **D)** 0.5

In [ ]:
lin = LinearRegression().fit(df[["balance"]], df["default"])
pred_lin = lin.predict(df[["balance"]])
print(f"Prediction for balance = $0: {lin.intercept_:.3f}")
print(f"Customers with a negative predicted 'probability': {(pred_lin < 0).mean():.0%}")

grid = pd.DataFrame({"balance": np.linspace(0, df["balance"].max(), 300)})
plt.scatter(df["balance"], df["default"], s=4, alpha=0.15, color=GREY, label="customers (0 = no, 1 = default)")
plt.plot(grid["balance"], lin.predict(grid), color=ORANGE, lw=2, label="linear regression")
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("balance ($)"); plt.ylabel("default"); plt.legend(loc="upper left")
plt.title("A straight line leaves [0, 1]")
plt.show()

With 0/1 coding, $E[Y \mid X] = \Pr(Y = 1 \mid X)$, so a linear fit *looks* reasonable, but it is not a
probability. With more than two classes it is worse: coding *stroke = 1, overdose = 2, seizure = 3* imposes an
order and equal distances that do not exist, and a different coding gives a different model.

## 2. Logistic regression

$$
p(X) = \Pr(Y = 1 \mid X) = \sigma(\beta_0 + \beta_1 X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}},
\qquad
\log\frac{p(X)}{1 - p(X)} = \beta_0 + \beta_1 X
$$

The output is always in $(0, 1)$. What is linear in $X$ is the **log-odds**.

In [ ]:
def coef_table(model):
    """Coefficients of a statsmodels Logit: estimate, std err, 95% CI, p-value and odds ratio e^beta."""
    ci = model.conf_int()
    return pd.DataFrame({"coef": model.params, "std err": model.bse, "CI 2.5%": ci[0], "CI 97.5%": ci[1],
                         "p-value": model.pvalues, "odds ratio": np.exp(model.params)}).round(5)

m_bal = smf.logit("default ~ balance", data=df).fit(disp=0)   # fitted by maximum likelihood
coef_table(m_bal)

In [ ]:
plt.scatter(df["balance"], df["default"], s=4, alpha=0.15, color=GREY)
plt.plot(grid["balance"], lin.predict(grid), color=ORANGE, lw=2, label="linear regression")
plt.plot(grid["balance"], m_bal.predict(grid), color=BLUE, lw=2, label="logistic regression")
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("balance ($)"); plt.ylabel("Pr(default)"); plt.legend(loc="upper left")
plt.show()

### 2.1 Interpreting the coefficients

**In pairs (calculator).** With $\hat\beta_0 = -10.651$ and $\hat\beta_1 = 0.0055$, compute the probability of
default for a balance of \$1,000 and of \$2,000. Then run the cell.

In [ ]:
b0, b1 = m_bal.params
for balance in [1000, 2000]:
    z = b0 + b1 * balance
    print(f"balance ${balance:,}:  z = {z:7.3f}   p = sigmoid(z) = {sigmoid(z):.4f}")

**Predict first.** $\hat\beta_1 = 0.0055$. If the balance increases by \$100, the **odds** of default…
**A)** increase by 0.55 · **B)** are multiplied by $e^{0.55} \approx 1.73$ · **C)** the probability increases by 0.55 ·
**D)** the probability is multiplied by 1.73

In [ ]:
odds = lambda p: p / (1 - p)
for start in [1000, 2000]:
    p_before, p_after = m_bal.predict(pd.DataFrame({"balance": [start, start + 100]}))
    print(f"${start:,} -> ${start + 100:,}:  probability {p_before:.4f} -> {p_after:.4f}"
          f"   odds x {odds(p_after) / odds(p_before):.3f}")
print(f"e^(100 * beta1) = {np.exp(100 * b1):.3f}")

The odds are always multiplied by the same factor; the change in **probability** depends on where you start.

### 2.2 Maximum likelihood and cross-entropy

We choose $\beta$ to make the observed labels as probable as possible:

$$
\ell(\beta) = \prod_{i=1}^{n} p(x_i)^{y_i}\bigl(1 - p(x_i)\bigr)^{1 - y_i}
\quad\Longrightarrow\quad
J(\beta) = -\frac{1}{n}\sum_{i=1}^{n}\Bigl[y_i \log p(x_i) + (1 - y_i)\log\bigl(1 - p(x_i)\bigr)\Bigr]
$$

Maximising $\ell$ is minimising $J$, the **cross-entropy** (or *log loss*). The log turns the product into a sum.

**Quick question.** A customer defaults. What loss does the model pay if it predicted $p = 0.9$? $p = 0.4$? $p = 0.001$?

In [ ]:
for p in [0.9, 0.4, 0.001]:
    print(f"predicted p = {p:<6}  loss = -log(p) = {-np.log(p):.3f}")

p_grid = np.linspace(0.001, 0.999, 300)
plt.plot(p_grid, -np.log(p_grid), color=BLUE, lw=2, label="true label y = 1:  -log p")
plt.plot(p_grid, -np.log(1 - p_grid), color=ORANGE, lw=2, label="true label y = 0:  -log(1 - p)")
plt.xlabel("predicted probability p"); plt.ylabel("loss"); plt.ylim(0, 7); plt.legend()
plt.title("Cross-entropy punishes confident mistakes")
plt.show()

The gradient has the same shape as in linear regression, $\nabla_\beta J = \frac{1}{n}X^\top(p - y)$, but $p$
depends on $\beta$ through the sigmoid, so setting it to zero has **no closed-form solution**. $J$ is convex, so an
iterative method finds the global minimum — that is next session (gradient descent). Let us check that the
gradient really is zero at the solution statsmodels found:

In [ ]:
X_design = np.column_stack([np.ones(len(df)), df["balance"]])   # column of ones + balance
y_vec = df["default"].to_numpy()

def cross_entropy(beta):
    p = sigmoid(X_design @ beta)
    return -np.mean(y_vec * np.log(p) + (1 - y_vec) * np.log(1 - p))

def gradient(beta):
    return X_design.T @ (sigmoid(X_design @ beta) - y_vec) / len(y_vec)

beta_hat = m_bal.params.to_numpy()
print("J at the statsmodels solution:", round(cross_entropy(beta_hat), 5), "  gradient:", gradient(beta_hat))
print("J at beta = (-10, 0.005):     ", round(cross_entropy(np.array([-10, 0.005])), 5))

> **scikit-learn note.** `LogisticRegression()` adds an L2 penalty by default (`C=1.0`), so its coefficients are
> not exactly the maximum-likelihood ones. Below we use `C=np.inf` (no penalty) to match the theory.

## 3. Confounding

Are students more likely to default?

In [ ]:
m_student = smf.logit("default ~ student", data=df).fit(disp=0)
display(coef_table(m_student))
df.groupby("student")["default"].mean().rename({0: "non-student", 1: "student"}).round(4)

**Predict first.** We add `balance` and `income` to the model. The `student` coefficient will be…
**A)** still positive · **B)** close to 0 · **C)** negative · **D)** it cannot change sign

In [ ]:
m_full = smf.logit("default ~ balance + income + student", data=df).fit(disp=0)
coef_table(m_full)

In [ ]:
bins = np.arange(0, 2751, 250)
rate = (df.groupby([pd.cut(df["balance"], bins), "student"], observed=True)["default"]
          .mean().unstack())
centres = [interval.mid for interval in rate.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for group, color, name in [(0, BLUE, "non-student"), (1, ORANGE, "student")]:
    axes[0].plot(centres, rate[group], "o-", color=color, label=name)
    axes[0].axhline(df.loc[df["student"] == group, "default"].mean(), color=color, ls="--", lw=1)
axes[0].set_xlabel("balance ($)"); axes[0].set_ylabel("default rate")
axes[0].set_title("At the same balance, students default less\n(dashed: overall rate of each group)")
axes[0].legend()
axes[1].boxplot([df.loc[df["student"] == g, "balance"] for g in (0, 1)])
axes[1].set_xticks([1, 2], ["non-student", "student"])
axes[1].set_ylabel("balance ($)"); axes[1].set_title("...but students carry larger balances")
plt.show()

Students have higher balances and higher balances mean more defaults, so *overall* students default more.
At the **same** balance they default less. A coefficient is always interpreted *holding the other features fixed* —
the same story as Newspaper in Session 3.

**Discuss (1 min).** Should a bank use "student" to decide who gets credit? Which of the two coefficients would it
use? We come back to this when we talk about fairness.

## 4. Evaluating a classifier

From here on we evaluate on data the model has not seen: 70% training, 30% test, keeping the proportion of
defaults equal in both (`stratify`). Anything we *choose* (like a threshold) is chosen with cross-validation on
the training set; the test set is used only to report results.

In [ ]:
features = ["balance", "student"]
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df["default"], test_size=0.3, stratify=df["default"], random_state=SEED)

def logistic(**kwargs):
    """Plain maximum-likelihood logistic regression; features standardised to help the solver."""
    return make_pipeline(StandardScaler(), LogisticRegression(C=np.inf, **kwargs))

model = logistic().fit(X_train, y_train)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)   # always predicts "no default"

print(f"Accuracy on the test set:  logistic model {model.score(X_test, y_test):.1%}"
      f"   |   always 'no default' {dummy.score(X_test, y_test):.1%}")

### 4.1 The confusion matrix

Back to today's question: a classifier that never predicts a default is almost as "accurate". Accuracy hides
*which* customers we get wrong.

In [ ]:
p_test = model.predict_proba(X_test)[:, 1]          # predicted Pr(default) for each test customer
pred_05 = (p_test >= 0.5).astype(int)               # the default decision rule: threshold 0.5

ConfusionMatrixDisplay.from_predictions(y_test, pred_05, display_labels=["no default", "default"],
                                        cmap="Blues", colorbar=False)
plt.grid(False); plt.title("Test set, threshold 0.5"); plt.show()

print(f"Recall (defaults detected):          {recall_score(y_test, pred_05):.1%}")
print(f"Precision (alerts that are correct): {precision_score(y_test, pred_05):.1%}")

| | Formula | Question it answers |
| --- | --- | --- |
| Recall (sensitivity, TPR) | TP / (TP + FN) | Of the real positives, how many do we catch? |
| Precision | TP / (TP + FP) | Of our alerts, how many are right? |
| Specificity | TN / (TN + FP) | Of the real negatives, how many do we leave alone? |
| F1 | 2 · P · R / (P + R) | Harmonic mean of precision and recall |

### 4.2 Base rates: an AI-text detector

A detector of AI-generated essays has 97% sensitivity and 97% specificity. Sounds great. But precision depends on
how common the positive class is:

In [ ]:
def precision_from_rates(sensitivity, specificity, prevalence):
    """Bayes' rule: fraction of flagged cases that are truly positive."""
    true_alarms = sensitivity * prevalence
    false_alarms = (1 - specificity) * (1 - prevalence)
    return true_alarms / (true_alarms + false_alarms)

for prevalence in [0.50, 0.20, 0.05, 0.01]:
    prec = precision_from_rates(0.97, 0.97, prevalence)
    print(f"{prevalence:4.0%} of essays written with AI -> {prec:.0%} of flagged students actually used AI")

If 5% of the essays are AI-written, more than a third of the flagged students are innocent. Rare positives make
precision fragile — exactly the situation of our defaults (3.3%).

### 4.3 The metric depends on the problem

**In pairs.** Model A is computed. Compute the metrics of model B by hand, then run the cell.
For colon-cancer screening (a positive means a colonoscopy), which metric matters most? And for a spam filter
(a positive goes to the spam folder)?

In [ ]:
def metrics_from_counts(tp, fn, fp, tn):
    precision, recall = tp / (tp + fp), tp / (tp + fn)
    return {"accuracy": (tp + tn) / (tp + fn + fp + tn), "precision": precision, "recall": recall,
            "specificity": tn / (tn + fp), "F1": 2 * precision * recall / (precision + recall)}

pd.DataFrame({"Model A": metrics_from_counts(tp=7, fn=3, fp=9, tn=81),
              "Model B": metrics_from_counts(tp=9, fn=1, fp=12, tn=78)}).T.round(3)

### 4.4 Choosing the threshold from costs

A missed defaulter costs the bank **10 times** more than a false alarm. Predicting "default" is worth it when the
expected cost of saying "no" is larger than the expected cost of saying "yes":

$$
p \cdot c_{FN} > (1 - p)\cdot c_{FP}
\quad\Longleftrightarrow\quad
p > \frac{c_{FP}}{c_{FP} + c_{FN}} = \frac{1}{11} \approx 0.091
$$

This assumes the probabilities are **calibrated** (Section 4.5). Let us check it on data the model has not been
trained on: out-of-fold probabilities from 5-fold cross-validation on the training set.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(logistic(), X_train, y_train, cv=cv, method="predict_proba")[:, 1]

C_FN, C_FP = 10, 1
def total_cost(y_true, y_pred):
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return C_FN * fn + C_FP * fp

rows = {}
for t in [0.5, 0.2, 0.1]:
    tn, fp, fn, tp = confusion_matrix(y_train, p_oof >= t).ravel()
    rows[t] = {"TP": tp, "FN": fn, "FP": fp, "recall": tp / (tp + fn), "precision": tp / (tp + fp),
               "cost": C_FN * fn + C_FP * fp}
pd.DataFrame(rows).T.rename_axis("threshold").astype({"TP": int, "FN": int, "FP": int, "cost": int}).round(3)

In [ ]:
thresholds = np.linspace(0.01, 0.9, 90)
costs = [total_cost(y_train.to_numpy(), (p_oof >= t).astype(int)) for t in thresholds]
best_t = thresholds[int(np.argmin(costs))]

plt.plot(thresholds, costs, color=BLUE, lw=2)
plt.axvline(1 / 11, color=ORANGE, ls="--", label="theory: 1/11 ≈ 0.091")
plt.axvline(best_t, color=GREY, ls=":", label=f"minimum on validation: {best_t:.2f}")
plt.axvline(0.5, color="black", lw=0.8, label="default threshold 0.5")
plt.xlabel("threshold"); plt.ylabel("total cost (units of one false alarm)"); plt.legend()
plt.title("Cost of the decisions as the threshold moves (cross-validation)")
plt.show()

scikit-learn (≥ 1.5) does this search for you with `TunedThresholdClassifierCV`: it chooses the threshold by
cross-validation on the training data. Only then do we look at the test set.

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV

neg_cost = make_scorer(lambda y_true, y_pred: -total_cost(np.asarray(y_true), y_pred))  # higher is better
tuned = TunedThresholdClassifierCV(logistic(), scoring=neg_cost, cv=cv).fit(X_train, y_train)
print(f"Threshold chosen by cross-validation: {tuned.best_threshold_:.3f}")

pd.Series({"threshold 0.5": total_cost(y_test.to_numpy(), pred_05),
           f"tuned threshold ({tuned.best_threshold_:.3f})": total_cost(y_test.to_numpy(), tuned.predict(X_test))},
          name="total cost on the test set")

**Back to today's question.** "Is 97.3% accuracy good?" was the wrong question. The right one is: what does each
kind of error cost, and which threshold minimises the total? Here a threshold near 0.1 cuts the cost by more than
40% while *lowering* accuracy.

### 4.5 Calibration: can we trust the probabilities?

A model is calibrated if, among the customers it gives $p \approx 0.2$, about 20% default. The cost rule above
needs it. A popular "fix" for imbalanced data, re-weighting the classes (`class_weight="balanced"`), breaks it:

In [ ]:
balanced = logistic(class_weight="balanced").fit(X_train, y_train)
p_balanced = balanced.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_predictions(y_test, p_test, n_bins=8, strategy="quantile", ax=ax,
                                    name="plain logistic", color=BLUE)
CalibrationDisplay.from_predictions(y_test, p_balanced, n_bins=8, strategy="quantile", ax=ax,
                                    name='class_weight="balanced"', color=ORANGE)
ax.set_title("Calibration on the test set"); plt.show()

pd.DataFrame({
    "mean predicted p": [p_test.mean(), p_balanced.mean()],
    "AUC": [roc_auc_score(y_test, p_test), roc_auc_score(y_test, p_balanced)],
    "Brier score": [brier_score_loss(y_test, p_test), brier_score_loss(y_test, p_balanced)],
    "true default rate": [y_test.mean(), y_test.mean()],
}, index=["plain logistic", 'class_weight="balanced"']).round(4)

Re-weighting ranks customers exactly as well (same AUC), but the average predicted probability jumps from 3.6% to
20% while the real default rate is 3.3%: the probabilities are no longer probabilities. If you need
a different trade-off between errors, **move the threshold**; keep the probabilities honest.

### 4.6 Errors by group

The same confusion matrix, computed separately for students and non-students (test set, tuned threshold):

In [ ]:
pred_tuned = tuned.predict(X_test)
rows = {}
for group, name in [(0, "non-student"), (1, "student")]:
    mask = (X_test["student"] == group).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_test[mask], pred_tuned[mask]).ravel()
    rows[name] = {"customers": mask.sum(), "default rate": y_test[mask].mean(), "flagged": pred_tuned[mask].mean(),
                  "TPR (recall)": tp / (tp + fn), "FPR": fp / (fp + tn), "precision": tp / (tp + fp)}
pd.DataFrame(rows).T.astype({"customers": int}).round(3)

**Discuss.** Students who would *not* default are flagged more often (higher FPR) than non-students. Is that
acceptable? Which of these numbers should be equal across groups for the model to be "fair"? (Machine Bias session.)

## 5. ROC and Precision-Recall curves

Every threshold gives one confusion matrix. Sweeping all thresholds gives a curve: **ROC** plots
TPR against FPR; **Precision-Recall** plots precision against recall. The dots are the thresholds 0.5 and the
tuned one.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, p_test)
prec, rec, _ = precision_recall_curve(y_test, p_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(fpr, tpr, color=BLUE, lw=2, label=f"logistic (AUC = {roc_auc_score(y_test, p_test):.3f})")
axes[0].plot([0, 1], [0, 1], color=GREY, ls=":", lw=1, label="random guess")
axes[1].plot(rec, prec, color=BLUE, lw=2, label=f"logistic (AP = {average_precision_score(y_test, p_test):.3f})")
axes[1].axhline(y_test.mean(), color=GREY, ls=":", lw=1, label="always 'default'")
for t, color in [(0.5, ORANGE), (tuned.best_threshold_, GREEN)]:
    tn, fp, fn, tp = confusion_matrix(y_test, p_test >= t).ravel()
    axes[0].plot(fp / (fp + tn), tp / (tp + fn), "o", color=color, ms=9, label=f"threshold {t:.2f}")
    axes[1].plot(tp / (tp + fn), tp / (tp + fp), "o", color=color, ms=9, label=f"threshold {t:.2f}")
axes[0].set(xlabel="false positive rate (FPR)", ylabel="true positive rate (TPR, recall)", title="ROC curve")
axes[1].set(xlabel="recall", ylabel="precision", title="Precision-Recall curve")
axes[0].legend(loc="lower right"); axes[1].legend(loc="upper right")
plt.show()

AUC is the probability that a random defaulter gets a higher score than a random non-defaulter.

**Predict first.** We add 9 extra copies of every non-defaulter to the test set, so defaults drop from 3.3% to
0.4%. The model does not change. Which curve changes? **A)** both · **B)** only ROC · **C)** only Precision-Recall · **D)** neither

In [ ]:
neg = (y_test == 0).to_numpy()
y_big = np.concatenate([y_test.to_numpy()] + [np.zeros(neg.sum(), dtype=int)] * 9)
p_big = np.concatenate([p_test] + [p_test[neg]] * 9)

pd.DataFrame({"default rate": [y_test.mean(), y_big.mean()],
              "ROC AUC": [roc_auc_score(y_test, p_test), roc_auc_score(y_big, p_big)],
              "average precision (PR)": [average_precision_score(y_test, p_test), average_precision_score(y_big, p_big)]},
             index=["original test set", "negatives x10"]).round(4)

TPR and FPR are computed *within* each true class, so ROC ignores how rare the positives are. Precision does not.
With rare positives, look at the Precision-Recall curve.

AUC only measures *ranking*. To judge the probabilities themselves use **log loss** (the cross-entropy of Section 2)
or the **Brier score** (mean squared error of the probabilities). Lower is better:

In [ ]:
p_prior = np.full(len(y_test), y_train.mean())   # always predict the training default rate
pd.DataFrame({"log loss": [log_loss(y_test, p_test), log_loss(y_test, p_prior)],
              "Brier score": [brier_score_loss(y_test, p_test), brier_score_loss(y_test, p_prior)]},
             index=["logistic", "always the base rate"]).round(4)

## 6. More than two classes: softmax

With $K$ classes, each class gets its own linear score $z_k = \beta_k^\top x$ and the **softmax** turns the scores
into probabilities:

$$
\Pr(Y = k \mid x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}
$$

With $K = 2$ it is the logistic function. It is also the output layer of every neural-network classifier — and of
large language models, which choose the next token among ~100,000 classes.

In [ ]:
def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max())          # subtracting the max does not change the result and avoids overflow
    return e / e.sum()

print("softmax([2.0, 1.0, -1.0]) =", softmax([2.0, 1.0, -1.0]).round(3))
print("K = 2: softmax([z, 0])[0] =", softmax([1.5, 0.0])[0].round(4), "   sigmoid(z) =", sigmoid(1.5).round(4))

**Handwritten digits** (8 × 8 images, 10 classes, included in scikit-learn). `LogisticRegression` uses the softmax
(multinomial) model automatically when there are more than two classes.

In [ ]:
digits = load_digits()
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    digits.data, digits.target, test_size=0.3, stratify=digits.target, random_state=SEED)

softmax_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xd_train, yd_train)
yd_pred = softmax_reg.predict(Xd_test)
print(f"Test accuracy: {accuracy_score(yd_test, yd_pred):.1%}")
print("Probabilities of the first test image sum to", softmax_reg.predict_proba(Xd_test[:1]).sum().round(6))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.2, 1]})
ConfusionMatrixDisplay.from_predictions(yd_test, yd_pred, ax=axes[0], cmap="Blues", colorbar=False)
axes[0].grid(False); axes[0].set_title("Confusion matrix (test)")
wrong = np.flatnonzero(yd_pred != yd_test)[:8]
axes[1].axis("off"); axes[1].set_title("Some mistakes: true -> predicted")
for k, i in enumerate(wrong):
    sub = axes[1].inset_axes([(k % 4) * 0.25, 0.5 - (k // 4) * 0.5, 0.22, 0.42])
    sub.imshow(Xd_test[i].reshape(8, 8), cmap="binary"); sub.set_xticks([]); sub.set_yticks([])
    sub.set_title(f"{yd_test[i]} -> {yd_pred[i]}", fontsize=10)
plt.show()

### 6.1 Accuracy vs macro averages

With imbalanced classes, accuracy is dominated by the large class. The **macro** average computes the metric for
each class and averages them, so every class counts the same. Two models that classify colon-cancer types A–D
(rows = true class, columns = predicted):

In [ ]:
model_a = np.array([[100, 80, 10, 10], [0, 9, 0, 1], [0, 1, 8, 1], [0, 1, 0, 9]])
model_b = np.array([[198, 2, 0, 0], [7, 1, 0, 2], [0, 8, 1, 1], [2, 3, 4, 1]])

def summary(cm):
    recall_per_class = np.diag(cm) / cm.sum(axis=1)
    return {"accuracy": np.trace(cm) / cm.sum(), "macro recall (balanced accuracy)": recall_per_class.mean(),
            **{f"recall type {c}": r for c, r in zip("ABCD", recall_per_class)}}

pd.DataFrame({"Model A": summary(model_a), "Model B": summary(model_b)}).T.round(3)

Model B wins on accuracy because it is good at the common type A, but it detects 1 in 10 cases of each rare type.
To diagnose rare types, model A is far better.

## 7. Optional: LDA, a generative alternative

Logistic regression models $\Pr(Y \mid X)$ directly (*discriminative*). Linear Discriminant Analysis models how $X$
is distributed in each class — a Gaussian with a shared covariance — and uses Bayes' rule to get $\Pr(Y \mid X)$
(*generative*). The decision boundary is also linear. See ISLR §4.4 for the derivation.

In [ ]:
lda = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()).fit(X_train, y_train)
p_lda = lda.predict_proba(X_test)[:, 1]
pd.DataFrame({"ROC AUC": [roc_auc_score(y_test, p_test), roc_auc_score(y_test, p_lda)],
              "log loss": [log_loss(y_test, p_test), log_loss(y_test, p_lda)]},
             index=["logistic regression", "LDA"]).round(4)

## 8. Wrap-up

**Exit ticket**

1. Why can't we fit logistic regression with a normal equation, as in Session 3?
2. A fraud model has 99.5% accuracy on data with 0.5% fraud. What do you check first?
3. Missing a defaulter costs 20 times a false alarm. Which threshold would you start from?
4. What was the least clear point today?

**Key ideas**

- Logistic regression models $\Pr(Y = 1 \mid X)$; a coefficient multiplies the **odds** by $e^{\beta}$, holding the other features fixed.
- It is fitted by maximum likelihood = minimising cross-entropy. No closed form → gradient descent (next session).
- A classifier outputs probabilities; the **decision** is a separate step: pick the threshold from the costs of
  each error, using validation data, never the test set.
- Accuracy hides imbalance. Use the confusion matrix, precision/recall, the PR curve with rare positives, and check
  the errors per group.
- Keep probabilities calibrated: change the threshold rather than re-weighting the classes.
- Softmax generalises the logistic function to $K$ classes.

## 9. Practice at home: a "5" detector

Build a binary classifier that answers "is this handwritten digit a 5?" on the `digits` data (about 10% positives).
If you want a bigger challenge, repeat it with MNIST (`fetch_openml("mnist_784")`, 70,000 images of 28 × 28).

In [ ]:
y5 = (digits.target == 5).astype(int)
print(f"{y5.mean():.1%} of the images are a 5")

**Task 1.** Split 70/30 (stratified, `random_state=SEED`). Compare the test accuracy of a `DummyClassifier` and of
`make_pipeline(StandardScaler(), LogisticRegression(C=np.inf, max_iter=5000))`. Show the confusion matrix of the
logistic model.

In [ ]:
X5_train, X5_test, y5_train, y5_test = train_test_split(
    digits.data, y5, test_size=0.3, stratify=y5, random_state=SEED)

dummy5 = DummyClassifier(strategy="most_frequent").fit(X5_train, y5_train)
detector = make_pipeline(StandardScaler(), LogisticRegression(C=np.inf, max_iter=5000)).fit(X5_train, y5_train)
print(f"Test accuracy: dummy {dummy5.score(X5_test, y5_test):.1%} | logistic {detector.score(X5_test, y5_test):.1%}")

ConfusionMatrixDisplay.from_predictions(y5_test, detector.predict(X5_test), display_labels=["not 5", "5"],
                                        cmap="Blues", colorbar=False)
plt.grid(False); plt.show()

**Task 2.** Report precision, recall and F1 at threshold 0.5, and the average precision (area under the PR curve).

In [ ]:
p5 = detector.predict_proba(X5_test)[:, 1]
pred5 = (p5 >= 0.5).astype(int)
print(f"precision {precision_score(y5_test, pred5):.3f} | recall {recall_score(y5_test, pred5):.3f} | "
      f"F1 {f1_score(y5_test, pred5):.3f} | average precision {average_precision_score(y5_test, p5):.3f}")

**Task 3.** Two ways to trade precision for recall. Compare on the test set:
(a) `class_weight="balanced"` with threshold 0.5, and (b) the plain model with the threshold chosen by
`TunedThresholdClassifierCV(..., scoring="f1")`. Report F1 and the Brier score of each. Which one would you deploy,
and why? Does re-weighting hurt calibration here as much as it did with Default? Why not?

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV

weighted5 = make_pipeline(StandardScaler(),
                          LogisticRegression(C=np.inf, max_iter=5000, class_weight="balanced")).fit(X5_train, y5_train)
tuned5 = TunedThresholdClassifierCV(
    make_pipeline(StandardScaler(), LogisticRegression(C=np.inf, max_iter=5000)),
    scoring="f1", cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)).fit(X5_train, y5_train)

pd.DataFrame({
    "threshold": [0.5, tuned5.best_threshold_],
    "F1": [f1_score(y5_test, weighted5.predict(X5_test)), f1_score(y5_test, tuned5.predict(X5_test))],
    "Brier score": [brier_score_loss(y5_test, weighted5.predict_proba(X5_test)[:, 1]),
                    brier_score_loss(y5_test, tuned5.predict_proba(X5_test)[:, 1])],
}, index=['(a) class_weight="balanced"', "(b) tuned threshold"]).round(3)

**Task 4.** Missing a 5 costs 5 times a false alarm. Which threshold does the cost rule of Section 4.4 give?
Is your model calibrated enough to trust it? (Hint: `CalibrationDisplay`.)

In [ ]:
print(f"Cost rule: predict '5' when p > 1 / (1 + 5) = {1 / 6:.3f}")
CalibrationDisplay.from_predictions(y5_test, p5, n_bins=10, strategy="uniform", name="plain logistic", color=BLUE)
plt.show()